# RAG Fundamentals + Observable Evaluation in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Merged notebook:** RAG Fundamentals + Observable RAG Evaluation  
**Runtime target:** SageMaker Jupyter Notebook  
**Model access:** Amazon Bedrock  
**Data source:** S3  
**Supported input files:** `.docx`, `.txt`, `.md`

This notebook combines the earlier **RAG Fundamentals** flow with the **Observable RAG Evaluation** flow into one classroom-ready lab.


## What learners will do

By the end of this notebook, learners will be able to:

- load `.docx`, `.txt`, and `.md` documents from S3
- compare chunking strategies and choose one for retrieval
- generate embeddings from a Bedrock embedding model available in the current region
- retrieve relevant chunks using in-memory similarity search
- optionally rerank retrieved chunks with an LLM
- generate grounded answers using Bedrock
- inspect retrieval traces, run metrics, and evaluation results in DataFrames
- improve the system during a mini-hack without saving any local artifacts

**Design choices in this merged version**
- no FAISS
- no local file saving
- no CSV ingestion
- no fallback corpus
- all observability stays in pandas DataFrames


In [3]:
# Uncomment only if your environment needs these libraries.
# %pip install -q boto3 pandas numpy python-docx


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Configuration and imports

Update the S3 bucket and prefix before running the notebook.

**Instructor cue:**  
Explain that this notebook is intentionally simple enough for the classroom:
- Bedrock for model access
- S3 for source documents
- pandas DataFrames for traceability
- no local vector store


In [31]:
from __future__ import annotations

import io
import json
import os
import re
from typing import Any

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from docx import Document as DocxDocument
from IPython.display import display

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# LLM model for answer generation and optional reranking.
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

# Leave this as None to auto-select an embedding model available in your region/account.
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# S3 bucket
S3_BUCKET = "rag-demo-docs-sri"
S3_PREFIX = "rag-docs/"

ALLOWED_EXTENSIONS = (".docx", ".txt", ".md")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 5
FINAL_K = 3

# Optional cap to make classroom runs faster. Set to None to embed every chunk.
MAX_CHUNKS_TO_EMBED = None

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
bedrock = boto3.client("bedrock", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("LLM model:", BEDROCK_LLM_MODEL_ID)
print("Requested embedding model:", BEDROCK_EMBEDDING_MODEL_ID or "AUTO-DISCOVER")
print("Allowed file types:", ALLOWED_EXTENSIONS)


AWS Region: us-east-1
LLM model: amazon.nova-lite-v1:0
Requested embedding model: amazon.titan-embed-text-v1
Allowed file types: ('.docx', '.txt', '.md')


## Step 2 — Discover available Bedrock models

This step helps learners see what is actually available in the current region and account.

**Discuss with learners:**
- model availability can vary by region and account
- a notebook should not blindly assume one embedding model is enabled
- discovery is often better than hardcoding


In [32]:
def validate_s3_inputs(bucket: str, prefix: str) -> None:
    if bucket == "replace-me" or prefix == "replace-me/rag_docs/":
        raise ValueError("Update S3_BUCKET and S3_PREFIX before running this notebook.")

def list_available_bedrock_models() -> pd.DataFrame:
    try:
        response = bedrock.list_foundation_models()
        model_summaries = response.get("modelSummaries", [])
    except Exception as e:
        print(f"Could not list Bedrock foundation models: {e}")
        return pd.DataFrame()

    rows = []
    for item in model_summaries:
        output_modalities = item.get("outputModalities", []) or []
        input_modalities = item.get("inputModalities", []) or []
        rows.append({
            "modelId": item.get("modelId"),
            "providerName": item.get("providerName"),
            "modelName": item.get("modelName"),
            "inputModalities": ", ".join(input_modalities) if isinstance(input_modalities, list) else str(input_modalities),
            "outputModalities": ", ".join(output_modalities) if isinstance(output_modalities, list) else str(output_modalities),
            "responseStreamingSupported": item.get("responseStreamingSupported"),
        })

    return pd.DataFrame(rows).sort_values(["providerName", "modelId"]).reset_index(drop=True)

all_models_df = list_available_bedrock_models()

embedding_models_df = all_models_df[
    all_models_df["modelId"].fillna("").str.startswith(("amazon.titan-embed", "cohere.embed"))
].reset_index(drop=True)

llm_models_df = all_models_df[
    all_models_df["outputModalities"].fillna("").str.contains("TEXT", case=False)
].reset_index(drop=True)

print("Embedding models visible in this region/account:")
display(embedding_models_df)

print("Text-capable models visible in this region/account:")
display(llm_models_df.head(30))


Embedding models visible in this region/account:


,modelId,providerName,modelName,inputModalities,outputModalities,responseStreamingSupported
0,amazon.titan-embed-g1-text-02,Amazon,Titan Text Embeddings v2,TEXT,EMBEDDING,None
1,amazon.titan-embed-image-v1,Amazon,Titan Multimodal Embeddings G1,"TEXT, IMAGE",EMBEDDING,None
2,amazon.titan-embed-image-v1:0,Amazon,Titan Multimodal Embeddings G1,"TEXT, IMAGE",EMBEDDING,None
3,amazon.titan-embed-text-v1,Amazon,Titan Embeddings G1 - Text,TEXT,EMBEDDING,False
4,amazon.titan-embed-text-v1:2:8k,Amazon,Titan Embeddings G1 - Text,TEXT,EMBEDDING,False
5,amazon.titan-embed-text-v2:0,Amazon,Titan Text Embeddings V2,TEXT,EMBEDDING,False
6,amazon.titan-embed-text-v2:0:8k,Amazon,Titan Text Embeddings V2,TEXT,EMBEDDING,False
7,cohere.embed-english-v3,Cohere,Embed English,TEXT,EMBEDDING,False
8,cohere.embed-english-v3:0:512,Cohere,Embed English,TEXT,EMBEDDING,False
9,cohere.embed-multilingual-v3,Cohere,Embed Multilingual,TEXT,EMBEDDING,False


Text-capable models visible in this region/account:


,modelId,providerName,modelName,inputModalities,outputModalities,responseStreamingSupported
0,ai21.jamba-1-5-large-v1:0,AI21 Labs,Jamba 1.5 Large,TEXT,TEXT,True
1,ai21.jamba-1-5-mini-v1:0,AI21 Labs,Jamba 1.5 Mini,TEXT,TEXT,True
2,amazon.nova-2-lite-v1:0,Amazon,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True
3,amazon.nova-2-lite-v1:0:256k,Amazon,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True
4,amazon.nova-2-sonic-v1:0,Amazon,Nova 2 Sonic,SPEECH,"SPEECH, TEXT",True
5,amazon.nova-lite-v1:0,Amazon,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True
6,amazon.nova-lite-v1:0:24k,Amazon,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True
7,amazon.nova-lite-v1:0:300k,Amazon,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True
8,amazon.nova-micro-v1:0,Amazon,Nova Micro,TEXT,TEXT,True
9,amazon.nova-micro-v1:0:128k,Amazon,Nova Micro,TEXT,TEXT,True


## Step 3 — Load `.docx`, `.txt`, and `.md` documents from S3

This notebook reads directly from S3 and keeps the loaded corpus in memory.

**Discuss with learners:**
- production systems usually separate storage from compute
- S3 is the document source of truth here
- loaders are format-specific because `.docx` and `.txt` are not parsed the same way


In [33]:
def list_document_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith(ALLOWED_EXTENSIONS):
                keys.append(key)

    return sorted(keys)

def read_s3_bytes(bucket: str, key: str) -> bytes:
    response = s3.get_object(Bucket=bucket, Key=key)
    return response["Body"].read()

def read_txt_or_md_from_s3(bucket: str, key: str) -> str:
    raw = read_s3_bytes(bucket, key)
    return raw.decode("utf-8", errors="ignore")

def read_docx_from_s3(bucket: str, key: str) -> str:
    raw = read_s3_bytes(bucket, key)
    stream = io.BytesIO(raw)
    doc = DocxDocument(stream)
    parts = []
    for p in doc.paragraphs:
        text = (p.text or "").strip()
        if text:
            parts.append(text)
    return "\n\n".join(parts)

def extract_text_from_s3(bucket: str, key: str) -> str:
    lower_key = key.lower()
    if lower_key.endswith(".docx"):
        return read_docx_from_s3(bucket, key)
    if lower_key.endswith(".txt") or lower_key.endswith(".md"):
        return read_txt_or_md_from_s3(bucket, key)
    raise ValueError(f"Unsupported file type for this notebook: {key}")

def infer_source_group(key: str, prefix: str) -> str:
    rel = key[len(prefix):] if key.startswith(prefix) else key
    rel = rel.strip("/")
    parts = rel.split("/")
    return parts[0] if len(parts) > 1 else "root"

validate_s3_inputs(S3_BUCKET, S3_PREFIX)

document_keys = list_document_keys(S3_BUCKET, S3_PREFIX)
print("Document count:", len(document_keys))
print(document_keys[:10])

raw_documents = []
for key in document_keys:
    text = extract_text_from_s3(S3_BUCKET, key).strip()
    if text:
        raw_documents.append({
            "source": key,
            "source_group": infer_source_group(key, S3_PREFIX),
            "file_type": key.split(".")[-1].lower(),
            "text": text,
            "char_len": len(text),
        })

docs_df = pd.DataFrame(raw_documents)
display(docs_df)
print("Loaded documents:", len(docs_df))


Document count: 9
['rag-docs/HR-POL-001_Employee_Onboarding_Policy.docx', 'rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx', 'rag-docs/HR-POL-003_Health_Benefits_Policy.docx', 'rag-docs/HR-POL-004_Training_and_Development_Policy.docx', 'rag-docs/HR-POL-005_Termination_and_Suspension_Policy.docx', 'rag-docs/HR-POL-006_Performance_Appraisal_Policy.docx', 'rag-docs/HR-POL-007_General_Work_Policy.docx', 'rag-docs/HR-POL-008_Leave_Policy.docx', 'rag-docs/HR-POL-009_Remote_Work_Policy.docx']


,source,source_group,file_type,text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,4869
1,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5318
2,rag-docs/HR-POL-003_Health_Benefits_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5159
3,rag-docs/HR-POL-004_Training_and_Development_P...,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5329
4,rag-docs/HR-POL-005_Termination_and_Suspension...,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5651
5,rag-docs/HR-POL-006_Performance_Appraisal_Poli...,root,docx,ABC Solutions Pvt. Ltd.\n\nDocument ID: HR-POL...,5418
6,rag-docs/HR-POL-007_General_Work_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4306
7,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,3708
8,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4137


Loaded documents: 9


## Step 4 — Compare chunking strategies

We compare three chunking approaches:

1. fixed-size chunking
2. fixed-size chunking with overlap
3. section-based chunking

**Instructor cue:**  
Use this section to help learners understand why chunking changes retrieval quality.


In [34]:
def basic_chunking(text: str, chunk_size: int = 500) -> list[str]:
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def chunk_with_overlap(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunk = text[i:i+chunk_size]
        if chunk:
            chunks.append(chunk)
    return chunks

def section_based_chunking(text: str, min_section_chars: int = 120) -> list[str]:
    sections = [sec.strip() for sec in re.split(r"\n\s*\n", text) if sec.strip()]
    return [sec for sec in sections if len(sec) >= min_section_chars]

if docs_df.empty:
    raise ValueError("No documents loaded from S3.")

sample_key = docs_df.iloc[0]["source"]
sample_text = docs_df.iloc[0]["text"]

chunking_comparison = {
    "basic": basic_chunking(sample_text),
    "overlap": chunk_with_overlap(sample_text),
    "section": section_based_chunking(sample_text),
}

comparison_rows = []
for method, chunks in chunking_comparison.items():
    comparison_rows.append({
        "method": method,
        "n_chunks": len(chunks),
        "first_chunk_preview": chunks[0][:300] if chunks else ""
    })

chunking_comparison_df = pd.DataFrame(comparison_rows)
display(chunking_comparison_df)


,method,n_chunks,first_chunk_preview
0,basic,10,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...
1,overlap,9,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...
2,section,14,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...


## Step 5 — Build chunk records with metadata

For the rest of the notebook we will use **overlap chunking**.  
Each chunk keeps metadata so that retrieval is explainable.

**Discuss with learners:**
- retrieval is better when chunks carry source metadata
- metadata helps filtering, debugging, and governance


In [35]:
chunk_rows = []

for row in docs_df.itertuples():
    chunks = chunk_with_overlap(row.text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    for i, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "source": row.source,
            "source_group": row.source_group,
            "file_type": row.file_type,
            "chunk_id": f"{row.source}::chunk_{i:03d}",
            "chunk_index": i,
            "chunk_text": chunk,
            "char_len": len(chunk),
        })

chunks_df = pd.DataFrame(chunk_rows)
display(chunks_df.head(10))
print("Total chunk rows:", len(chunks_df))


,source,source_group,file_type,chunk_id,chunk_index,chunk_text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,1,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,700
1,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2,"nology Act, 2000, the Employees’ Provident Fun...",700
2,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,3,Chief Information Security Officer (CISO) and...,700
3,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,4,1 Offer and Pre-Joining Formalities\n4.1.1 Fin...,700
4,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,5,V shall be initiated on DoJ and completed with...,700
5,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,6,of a minimum of 16 hours of classroom sessions...,700
6,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,7,", BGV, or security clearance within prescribed...",700
7,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,8,"r HR-POL-009, Clause 3.1.\n7.4 Information Sec...",700
8,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,9,"cuments, or violation of information security ...",229
9,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,root,docx,rag-docs/HR-POL-002_Equal_Opportunity_Policy.d...,1,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,700


Total chunk rows: 81


## Step 6 — Discover and use a Bedrock embedding model

This notebook does not assume a specific embedding model is enabled.  
It first discovers what is available, then uses a supported payload shape for that model.

**Instructor cue:**  
Show learners that API payloads can differ by model provider.


In [36]:
def resolve_embedding_model(preferred_model_id: str | None = None) -> str:
    available_ids = embedding_models_df["modelId"].dropna().tolist() if len(embedding_models_df) else []

    if preferred_model_id:
        if preferred_model_id not in available_ids:
            raise ValueError(
                f"Requested embedding model '{preferred_model_id}' is not listed as available in region {AWS_REGION}. "
                f"Available models: {available_ids}"
            )
        return preferred_model_id

    priority_order = [
        "amazon.titan-embed-text-v2:0",
        "amazon.titan-embed-text-v1",
        "cohere.embed-english-v3",
        "cohere.embed-multilingual-v3",
    ]

    for model_id in priority_order:
        if model_id in available_ids:
            return model_id

    if available_ids:
        return available_ids[0]

    raise ValueError(
        f"No Bedrock embedding models were listed in region {AWS_REGION}. "
        "Enable an embedding model in this account/region and try again."
    )

RESOLVED_EMBEDDING_MODEL_ID = resolve_embedding_model(BEDROCK_EMBEDDING_MODEL_ID)
print("Resolved embedding model:", RESOLVED_EMBEDDING_MODEL_ID)

def build_embedding_payload(text: str, model_id: str, purpose: str = "document") -> dict[str, Any]:
    model_id = model_id.lower()

    if "amazon.titan-embed-text-v2" in model_id:
        return {"inputText": text}

    if "amazon.titan-embed-text-v1" in model_id:
        return {"inputText": text}

    if model_id.startswith("cohere.embed"):
        input_type = "search_query" if purpose == "query" else "search_document"
        return {
            "texts": [text],
            "input_type": input_type,
            "truncate": "END",
        }

    raise ValueError(
        f"Unsupported embedding model format for helper: {model_id}. "
        "Use a Titan or Cohere embedding model available in Bedrock."
    )

def parse_embedding_response(model_response: dict[str, Any]) -> list[float]:
    if "embedding" in model_response and isinstance(model_response["embedding"], list):
        return model_response["embedding"]

    if "embeddings" in model_response:
        embeddings = model_response["embeddings"]
        if isinstance(embeddings, list):
            if embeddings and isinstance(embeddings[0], list):
                return embeddings[0]
            return embeddings

    if "embeddingsByType" in model_response:
        embeddings_by_type = model_response["embeddingsByType"]
        if isinstance(embeddings_by_type, dict):
            for value in embeddings_by_type.values():
                if isinstance(value, list):
                    if value and isinstance(value[0], list):
                        return value[0]
                    return value

    raise ValueError(f"Could not parse embedding response keys: {list(model_response.keys())}")

def l2_normalize(vector: list[float] | np.ndarray) -> np.ndarray:
    arr = np.array(vector, dtype="float32")
    norm = np.linalg.norm(arr)
    return arr if norm == 0 else arr / norm

def get_text_embedding(text: str, model_id: str = RESOLVED_EMBEDDING_MODEL_ID, purpose: str = "document") -> list[float]:
    payload = build_embedding_payload(text=text, model_id=model_id, purpose=purpose)

    try:
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps(payload),
            accept="application/json",
            contentType="application/json",
        )
    except ClientError as e:
        raise RuntimeError(f"Bedrock embedding call failed for model {model_id}: {e}") from e

    model_response = json.loads(response["body"].read())
    return parse_embedding_response(model_response)

if MAX_CHUNKS_TO_EMBED is not None:
    working_chunks_df = chunks_df.head(MAX_CHUNKS_TO_EMBED).copy().reset_index(drop=True)
else:
    working_chunks_df = chunks_df.copy().reset_index(drop=True)

working_chunks_df["embedding"] = working_chunks_df["chunk_text"].apply(
    lambda text: get_text_embedding(text, purpose="document")
)
working_chunks_df["embedding_dim"] = working_chunks_df["embedding"].apply(len)

embedding_matrix = np.vstack([
    l2_normalize(vec) for vec in working_chunks_df["embedding"].tolist()
]).astype("float32")

display(
    working_chunks_df.drop(columns=["embedding"]).head(10)
)
print("Embedded chunk count:", len(working_chunks_df))
print("Embedding dimension:", int(working_chunks_df.iloc[0]["embedding_dim"]) if len(working_chunks_df) else 0)


Resolved embedding model: amazon.titan-embed-text-v1


,source,source_group,file_type,chunk_id,chunk_index,chunk_text,char_len,embedding_dim
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,1,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,700,1536
1,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2,"nology Act, 2000, the Employees’ Provident Fun...",700,1536
2,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,3,Chief Information Security Officer (CISO) and...,700,1536
3,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,4,1 Offer and Pre-Joining Formalities\n4.1.1 Fin...,700,1536
4,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,5,V shall be initiated on DoJ and completed with...,700,1536
5,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,6,of a minimum of 16 hours of classroom sessions...,700,1536
6,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,7,", BGV, or security clearance within prescribed...",700,1536
7,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,8,"r HR-POL-009, Clause 3.1.\n7.4 Information Sec...",700,1536
8,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,9,"cuments, or violation of information security ...",229,1536
9,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,root,docx,rag-docs/HR-POL-002_Equal_Opportunity_Policy.d...,1,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,700,1536


Embedded chunk count: 81
Embedding dimension: 1536


## Step 7 — Retrieval by similarity

This section performs in-memory similarity search using embeddings and cosine-style scoring.

**Discuss with learners:**
- this is enough for classroom-scale data
- production systems may use a vector database, but the retrieval logic is still conceptually the same
- metadata filters can change the final result set


In [37]:
def retrieve(query: str, top_k: int = TOP_K, source_group: str | None = None) -> tuple[pd.DataFrame, str]:
    q_emb = l2_normalize(get_text_embedding(query, purpose="query"))

    candidate_df = working_chunks_df.copy()
    if source_group:
        candidate_df = candidate_df[candidate_df["source_group"] == source_group].reset_index(drop=True)

    if candidate_df.empty:
        raise ValueError(f"No chunks available for source_group={source_group!r}")

    candidate_matrix = np.vstack([
        l2_normalize(vec) for vec in candidate_df["embedding"].tolist()
    ]).astype("float32")

    scores = candidate_matrix @ q_emb
    ranked_idxs = np.argsort(-scores)[:top_k]

    rows = []
    for rank, idx_ in enumerate(ranked_idxs, start=1):
        row = candidate_df.iloc[int(idx_)]
        rows.append({
            "query": query,
            "rank": rank,
            "score": float(scores[int(idx_)]),
            "source": row["source"],
            "source_group": row["source_group"],
            "file_type": row["file_type"],
            "chunk_id": row["chunk_id"],
            "chunk_text": row["chunk_text"],
        })

    retrieval_df = pd.DataFrame(rows)
    context = "\n\n".join(
        [f"[{r.rank}] [Source: {r.source}]\n{r.chunk_text}" for r in retrieval_df.itertuples()]
    )
    return retrieval_df, context

test_retrieval_df, test_context = retrieve("What does the policy say about leave?", top_k=TOP_K)
display(test_retrieval_df)
print(test_context[:1500])


,query,rank,score,source,source_group,file_type,chunk_id,chunk_text
0,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,root,docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...
1,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...
3,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ..."
4,What does the policy say about leave?,5,0.589947,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,"roductivity, and compliance linkage.\n9.3\n\nH..."


[1] [Source: rag-docs/HR-POL-007_General_Work_Policy.docx]
 restricted to authorized personnel with valid identity cards.
6.3 Violation of safety or security protocols shall be treated as serious misconduct.

7. Leave, Absence, and Punctuality

7.1 Leave shall be availed only after approval in accordance with Leave Policy (HR-POL-008).
7.2 Repeated late coming or early departure beyond three instances in a month may attract disciplinary counselling and notation in performance records under HR-POL-006.

8. Roles and Responsibilities

Role

Key Responsibilities

9. Cross-Policy Dependencies

9.1

Leave Policy (HR-POL-008):

Governs all forms of authorized and unauthorized absence.
9.2

Performance Appraisal Policy (HR-POL-006):

Attendance and conduct

[2] [Source: rag-docs/HR-POL-008_Leave_Policy.docx]
ABC Solutions Pvt. Ltd.

ABC Solutions Pvt. Ltd.
Document ID:

HR-POL-008

Version:

1.0

Effective Date:

January 1, 2026

Title:

Leave Policy

Classification:

Confidential – Internal 

## Step 8 — Optional reranking with an LLM

Initial retrieval is based on embedding similarity.  
Reranking asks the LLM to reorder the retrieved chunks by relevance.

**Instructor cue:**  
Explain the difference:
- retrieval finds semantically close chunks
- reranking tries to improve the order for the current query


In [38]:
def rerank_results_llm(query: str, retrieval_df: pd.DataFrame, final_k: int = FINAL_K) -> pd.DataFrame:
    if retrieval_df.empty:
        return retrieval_df.copy()

    docs = ""
    for i, row in enumerate(retrieval_df.itertuples(index=False), start=0):
        docs += f"\nDocument {i}:\n{row.chunk_text}\n"

    prompt = f'''
You are a retrieval reranking assistant.

Given a query and multiple documents, return ONLY a Python-style list
of the most relevant document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Return only something like:
[2, 0, 1]
'''.strip()

    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": 0}
    )

    output = response["output"]["message"]["content"][0]["text"]
    indices = list(map(int, re.findall(r"\d+", output)))

    if not indices:
        return retrieval_df.head(final_k).copy().reset_index(drop=True)

    reranked_rows = []
    used = set()
    for idx_ in indices:
        if 0 <= idx_ < len(retrieval_df) and idx_ not in used:
            reranked_rows.append(retrieval_df.iloc[idx_].to_dict())
            used.add(idx_)

    reranked_df = pd.DataFrame(reranked_rows).head(final_k).reset_index(drop=True)
    if reranked_df.empty:
        reranked_df = retrieval_df.head(final_k).copy().reset_index(drop=True)

    reranked_df["rerank_position"] = range(1, len(reranked_df) + 1)
    return reranked_df

candidate_df, _ = retrieve("What is eligibility for remote work?", top_k=TOP_K)
reranked_df_preview = rerank_results_llm("What is eligibility for remote work?", candidate_df, final_k=FINAL_K)
display(reranked_df_preview)


,query,rank,score,source,source_group,file_type,chunk_id,chunk_text,rerank_position
0,What is eligibility for remote work?,3,0.668556,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,ays per week remote work: BUH approval.\n4.2.2...,1
1,What is eligibility for remote work?,2,0.712409,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,ith applicable labour laws and the Information...,2
2,What is eligibility for remote work?,1,0.728857,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,rk:\n\nA combination of on-site and remote wor...,3


## Step 9 — Build grounded answers with Bedrock

This is the final RAG step:
1. retrieve
2. optionally rerank
3. build context
4. generate a grounded answer

**Discuss with learners:**
- the answer quality depends on the context quality
- RAG does not remove hallucination risk, but it reduces it when context is good


In [39]:
def call_bedrock_llm(prompt: str, model_id: str = BEDROCK_LLM_MODEL_ID, max_tokens: int = 300) -> str:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]

def build_context_from_df(df: pd.DataFrame) -> str:
    return "\n\n".join(
        [f"[Source: {row.source}]\n{row.chunk_text}" for row in df.itertuples()]
    )

def answer_with_rag(query: str,
                    retrieve_k: int = TOP_K,
                    final_k: int = FINAL_K,
                    source_group: str | None = None,
                    use_reranker: bool = True) -> dict[str, Any]:
    retrieval_df, _ = retrieve(query=query, top_k=retrieve_k, source_group=source_group)

    if use_reranker:
        final_context_df = rerank_results_llm(query=query, retrieval_df=retrieval_df, final_k=final_k)
    else:
        final_context_df = retrieval_df.head(final_k).copy().reset_index(drop=True)

    context = build_context_from_df(final_context_df)

    prompt = f'''
You are an enterprise RAG assistant.

Answer ONLY from the provided context.
If the answer is not present, say: "I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
'''.strip()

    answer = call_bedrock_llm(prompt)

    run_metrics_df = pd.DataFrame([{
        "query": query,
        "retrieve_k": retrieve_k,
        "final_k": final_k,
        "use_reranker": use_reranker,
        "source_group_filter": source_group,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "n_documents": len(docs_df),
        "n_total_chunks": len(chunks_df),
        "n_embedded_chunks": len(working_chunks_df),
        "timestamp_utc": pd.Timestamp.utcnow(),
    }])

    return {
        "retrieval_df": retrieval_df,
        "final_context_df": final_context_df,
        "prompt": prompt,
        "answer": answer,
        "run_metrics_df": run_metrics_df,
    }

demo_run = answer_with_rag("What does the policy say about leave?", use_reranker=True)

print("ANSWER\n")
print(demo_run["answer"])

print("\nInitial retrieval")
display(demo_run["retrieval_df"])

print("\nFinal context after reranking")
display(demo_run["final_context_df"])

print("\nRun metrics")
display(demo_run["run_metrics_df"])


ANSWER

The policy defines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. The types of leave include Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave. The policy applies to all confirmed employees across all grades and locations, and it is administered by the Human Resources Department. Leave records must be maintained in the HRIS and retained for a minimum of 7 years. Annual audits shall verify compliance with accrual, carry-forward, and encashment rules. Any violation of this policy shall be subject to disciplinary action.

Initial retrieval


,query,rank,score,source,source_group,file_type,chunk_id,chunk_text
0,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,root,docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...
1,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...
3,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ..."
4,What does the policy say about leave?,5,0.589947,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,"roductivity, and compliance linkage.\n9.3\n\nH..."



Final context after reranking


,query,rank,score,source,source_group,file_type,chunk_id,chunk_text,rerank_position
0,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,1
1,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ...",2
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...,3



Run metrics


,query,retrieve_k,final_k,use_reranker,source_group_filter,chunk_size,chunk_overlap,n_documents,n_total_chunks,n_embedded_chunks,timestamp_utc
0,What does the policy say about leave?,5,3,True,None,700,120,9,81,81,2026-04-17 13:00:46.781944+00:00


## Step 10 — Lightweight evaluation in DataFrames

This section keeps evaluation simple and classroom-friendly.

**Discuss with learners:**
- evaluation does not need to be perfect to be useful
- even a lightweight metric can reveal whether retrieval changes help or hurt
- for classroom use, DataFrames are enough for visibility


In [40]:
def normalize_text(s: str) -> list[str]:
    return re.findall(r"\w+", (s or "").lower())

def overlap_f1(pred: str, ref: str) -> float:
    p = normalize_text(pred)
    r = normalize_text(ref)
    if not p or not r:
        return 0.0

    p_set, r_set = set(p), set(r)
    inter = len(p_set & r_set)
    precision = inter / len(p_set) if p_set else 0.0
    recall = inter / len(r_set) if r_set else 0.0

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

golden_df = pd.DataFrame([
    {
        "question": "What does the policy say about leave?",
        "reference_answer": "The answer should come from the leave policy content in the retrieved documents."
    },
    {
        "question": "What are the rules for remote work?",
        "reference_answer": "The answer should come from the remote work content in the retrieved documents."
    },
])

eval_rows = []
for row in golden_df.itertuples():
    run = answer_with_rag(row.question, use_reranker=True)
    top_source = run["final_context_df"].iloc[0]["source"] if len(run["final_context_df"]) else None

    eval_rows.append({
        "question": row.question,
        "prediction": run["answer"],
        "reference_answer": row.reference_answer,
        "overlap_f1": overlap_f1(run["answer"], row.reference_answer),
        "top_source": top_source,
        "n_context_chunks": len(run["final_context_df"]),
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)
print("Average overlap_f1:", round(eval_df["overlap_f1"].mean(), 4))


,question,prediction,reference_answer,overlap_f1,top_source,n_context_chunks
0,What does the policy say about leave?,The Leave Policy (HR-POL-008) defines the type...,The answer should come from the leave policy c...,0.095238,rag-docs/HR-POL-008_Leave_Policy.docx,3
1,What are the rules for remote work?,The rules for remote work are as follows:\n\n1...,The answer should come from the remote work co...,0.075758,rag-docs/HR-POL-009_Remote_Work_Policy.docx,3


Average overlap_f1: 0.0855


## Step 11 — What to inspect in this notebook

At this point learners should inspect four things:

1. **documents table** — what was loaded from S3  
2. **chunk table** — how the source was split  
3. **retrieval table** — which chunks were selected  
4. **evaluation table** — whether the answer is improving

**Instructor cue:**  
This is a good place to pause and ask learners what they would change first:
- chunk size
- overlap
- top_k
- reranking on/off
- metadata filters


In [41]:
print("Loaded documents")
display(docs_df)

print("\nChunk sample")
display(chunks_df.head(10))

print("\nLatest retrieval sample")
display(demo_run["retrieval_df"])

print("\nLatest evaluation results")
display(eval_df)


Loaded documents


,source,source_group,file_type,text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,4869
1,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5318
2,rag-docs/HR-POL-003_Health_Benefits_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5159
3,rag-docs/HR-POL-004_Training_and_Development_P...,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5329
4,rag-docs/HR-POL-005_Termination_and_Suspension...,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,5651
5,rag-docs/HR-POL-006_Performance_Appraisal_Poli...,root,docx,ABC Solutions Pvt. Ltd.\n\nDocument ID: HR-POL...,5418
6,rag-docs/HR-POL-007_General_Work_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4306
7,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,3708
8,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,4137



Chunk sample


,source,source_group,file_type,chunk_id,chunk_index,chunk_text,char_len
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,1,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,700
1,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2,"nology Act, 2000, the Employees’ Provident Fun...",700
2,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,3,Chief Information Security Officer (CISO) and...,700
3,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,4,1 Offer and Pre-Joining Formalities\n4.1.1 Fin...,700
4,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,5,V shall be initiated on DoJ and completed with...,700
5,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,6,of a minimum of 16 hours of classroom sessions...,700
6,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,7,", BGV, or security clearance within prescribed...",700
7,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,8,"r HR-POL-009, Clause 3.1.\n7.4 Information Sec...",700
8,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,root,docx,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,9,"cuments, or violation of information security ...",229
9,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx,root,docx,rag-docs/HR-POL-002_Equal_Opportunity_Policy.d...,1,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...,700



Latest retrieval sample


,query,rank,score,source,source_group,file_type,chunk_id,chunk_text
0,What does the policy say about leave?,1,0.671852,rag-docs/HR-POL-007_General_Work_Policy.docx,root,docx,rag-docs/HR-POL-007_General_Work_Policy.docx::...,restricted to authorized personnel with valid...
1,What does the policy say about leave?,2,0.626138,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_001,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...
2,What does the policy say about leave?,3,0.623699,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_006,sive unauthorized absence shall impact complia...
3,What does the policy say about leave?,4,0.597447,rag-docs/HR-POL-008_Leave_Policy.docx,root,docx,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_002,"ory compliance, while providing adequate rest ..."
4,What does the policy say about leave?,5,0.589947,rag-docs/HR-POL-009_Remote_Work_Policy.docx,root,docx,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,"roductivity, and compliance linkage.\n9.3\n\nH..."



Latest evaluation results


,question,prediction,reference_answer,overlap_f1,top_source,n_context_chunks
0,What does the policy say about leave?,The Leave Policy (HR-POL-008) defines the type...,The answer should come from the leave policy c...,0.095238,rag-docs/HR-POL-008_Leave_Policy.docx,3
1,What are the rules for remote work?,The rules for remote work are as follows:\n\n1...,The answer should come from the remote work co...,0.075758,rag-docs/HR-POL-009_Remote_Work_Policy.docx,3


## Mini-hack — Improve the RAG system

Ask learners to work in pairs and try one change at a time.

### Suggested learner tasks
1. increase or decrease `CHUNK_SIZE`
2. increase or decrease `CHUNK_OVERLAP`
3. change `TOP_K` or `FINAL_K`
4. turn reranking off and compare the answer
5. use `source_group` filtering if your S3 prefix contains multiple folders
6. add one more evaluation question to `golden_df`

### Learner work area
- note what you changed
- record whether retrieval improved
- record whether the answer became more grounded
- compare the evaluation DataFrame before and after your change


## Wrap-up

Learners should now understand:

- how to load a small corpus from S3 into a SageMaker notebook
- how chunking choices affect retrieval
- how Bedrock embeddings can be used without FAISS
- how to retrieve and rerank chunks before answer generation
- how to keep observability simple with DataFrames
- how to run a lightweight evaluation loop inside the notebook

This merged notebook is a cleaner classroom flow because it covers:
**core RAG + observability + evaluation** in one place.
